In [2]:
# All raw OULAD CSVs land here
BASE_PATH = "Files/uploads/oulad"

# Bronze Delta tables will be written here
BRONZE_PATH = "Files/bronze/batch"

# Convenience: table names mapped to filenames
OULAD_FILES = {
    "bronze_courses":             "courses.csv",
    "bronze_assessments":         "assessments.csv",
    "bronze_vle":                 "vle.csv",
    "bronze_student_info":        "studentInfo.csv",
    "bronze_student_registration":"studentRegistration.csv",
    "bronze_student_assessment":  "studentAssessment.csv",
    "bronze_student_vle":         "studentVle.csv",
}

StatementMeta(, 1eacd33d-eb98-4912-ac51-d91e630457b5, 4, Finished, Available, Finished, False)

In [4]:
from pyspark.sql import functions as F

results = {}

for table_name, filename in OULAD_FILES.items():
    file_path = f"{BASE_PATH}/{filename}"
    
    df = spark.read.csv(
        file_path,
        header=True,
        inferSchema=True,
        nullValue="?",         # OULAD uses "?" for missing values
    )
    
    # Add ingestion metadata
    df = df.withColumn("_ingested_at", F.current_timestamp()) \
           .withColumn("_source_file", F.lit(filename))
    
    df.write.format("delta") \
      .mode("overwrite") \
      .option("overwriteSchema", "true") \
      .saveAsTable(table_name)
    
    count = df.count()
    results[table_name] = count
    print(f"{table_name}: {count:,} rows")

StatementMeta(, 1eacd33d-eb98-4912-ac51-d91e630457b5, 6, Finished, Available, Finished, False)

bronze_courses: 22 rows
bronze_assessments: 206 rows
bronze_vle: 6,364 rows
bronze_student_info: 32,593 rows
bronze_student_registration: 32,593 rows
bronze_student_assessment: 173,912 rows
bronze_student_vle: 10,655,280 rows


In [7]:
expected_minimums = {
    "bronze_courses":              20,
    "bronze_assessments":         200,
    "bronze_vle":                5000,
    "bronze_student_info":       30000,
    "bronze_student_registration":30000,
    "bronze_student_assessment": 150000,
    "bronze_student_vle":      5000000,
}

print("\n--- Bronze Validation ---")
all_passed = True
for table, min_rows in expected_minimums.items():
    actual = results[table]
    status = "PASS" if actual >= min_rows else "FAIL"
    if actual < min_rows:
        all_passed = False
    print(f"{status}  {table}: {actual:,} rows (expected ≥ {min_rows:,})")

print("\nAll checks passed." if all_passed else "\nOne or more checks failed — review above.")

StatementMeta(, 1eacd33d-eb98-4912-ac51-d91e630457b5, 9, Finished, Available, Finished, False)


--- Bronze Validation ---
PASS  bronze_courses: 22 rows (expected ≥ 20)
PASS  bronze_assessments: 206 rows (expected ≥ 200)
PASS  bronze_vle: 6,364 rows (expected ≥ 5,000)
PASS  bronze_student_info: 32,593 rows (expected ≥ 30,000)
PASS  bronze_student_registration: 32,593 rows (expected ≥ 30,000)
PASS  bronze_student_assessment: 173,912 rows (expected ≥ 150,000)
PASS  bronze_student_vle: 10,655,280 rows (expected ≥ 5,000,000)

All checks passed.


In [8]:
critical_checks = {
    "bronze_student_info":        ["id_student", "final_result"],
    "bronze_student_assessment":  ["id_student", "id_assessment", "score"],
    "bronze_student_vle":         ["id_student", "id_site", "date", "sum_click"],
    "bronze_student_registration":["id_student", "code_module", "code_presentation"],
}

print("\n--- Null Checks on Critical Columns ---")
for table, cols in critical_checks.items():
    df = spark.table(table)
    for col in cols:
        null_count = df.filter(F.col(col).isNull()).count()
        status = "✅" if null_count == 0 else f"⚠️  {null_count:,} nulls"
        print(f"  {status}  {table}.{col}")

StatementMeta(, 1eacd33d-eb98-4912-ac51-d91e630457b5, 10, Finished, Available, Finished, False)


--- Null Checks on Critical Columns ---
  ✅  bronze_student_info.id_student
  ✅  bronze_student_info.final_result
  ✅  bronze_student_assessment.id_student
  ✅  bronze_student_assessment.id_assessment
  ⚠️  173 nulls  bronze_student_assessment.score
  ✅  bronze_student_vle.id_student
  ✅  bronze_student_vle.id_site
  ✅  bronze_student_vle.date
  ✅  bronze_student_vle.sum_click
  ✅  bronze_student_registration.id_student
  ✅  bronze_student_registration.code_module
  ✅  bronze_student_registration.code_presentation


In [9]:
print("\n--- Bronze Table Schemas ---")
for table_name in OULAD_FILES.keys():
    print(f"\n📋 {table_name}")
    spark.table(table_name).printSchema()

StatementMeta(, 1eacd33d-eb98-4912-ac51-d91e630457b5, 11, Finished, Available, Finished, False)


--- Bronze Table Schemas ---

📋 bronze_courses
root
 |-- code_module: string (nullable = true)
 |-- code_presentation: string (nullable = true)
 |-- module_presentation_length: integer (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)


📋 bronze_assessments
root
 |-- code_module: string (nullable = true)
 |-- code_presentation: string (nullable = true)
 |-- id_assessment: integer (nullable = true)
 |-- assessment_type: string (nullable = true)
 |-- date: integer (nullable = true)
 |-- weight: double (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)


📋 bronze_vle
root
 |-- id_site: integer (nullable = true)
 |-- code_module: string (nullable = true)
 |-- code_presentation: string (nullable = true)
 |-- activity_type: string (nullable = true)
 |-- week_from: integer (nullable = true)
 |-- week_to: integer (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
